# Tome 4, Chapter 11: Asymptotics (Python companion)

Exercises first, solutions at the end. We use a divergent series at its best, sum with Euler–Maclaurin (and continue $\zeta$ with it), test the error bounds of Stirling's series, check Laplace's method and the saddle point, measure quicksort against its asymptotic expansion, and test the transfer theorem on graphs, paths and partitions. Requirements: mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P11.1 (★) A divergent series at its best.** For Euler's function $f(x)=\int_0^\infty\frac{e^{-t}}{1+xt}\,dt$ (Proposition 11.3), tabulate the error of the partial sums $\sum_{k<K}(-1)^kk!\,x^k$ against $K$ for $x=0.1$ and $x=0.05$, find the best $K$, and compare the best error with $\sqrt{2\pi/x}\,e^{-1/x}$. Then use the sign of the error to bracket $f(0.1)$ between two partial sums.

**P11.2 (★★) Euler–Maclaurin at work.** Implement Proposition 11.7 (without the remainder) and compute $\zeta(s)$ at $s=3$, $2$, $\frac12+14.134725i$, $-2.5+3i$ and $-1.5$ with $N=10$ and six Bernoulli terms. Then, with exact fractions and $N=1$, check that it gives $\zeta(-n)=-\frac{B_{n+1}}{n+1}$ (Theorem 10.10) for $n\le20$. Finally test Corollary 11.6: is the error of the expansion of $H_n$ really smaller than the first omitted term, and of the same sign?

**P11.3 (★★) Stirling's series.** Check numerically that $0<\rho_m(x)/T_m(x)<1$ (Theorem 11.8) for $x=0.5,1,\dots,10$ and $m=1,\dots,11$. For $x=1,2,4,8$ find the smallest term $|T_m(x)|$ and compare it with $\frac{e^{-2\pi x}}{\pi\sqrt x}$. Then write a function computing $\ln\Gamma(x)$ for any $x>0$ to about $17$ digits: shift $x$ up by $10$ with $\Gamma(x+1)=x\Gamma(x)$, then apply the series.

**P11.4 (★) Laplace and the saddle point.** Check $n!=n^{n+1}\int_0^\infty e^{n(\ln u-u)}\,du$ numerically, and compare $n!$ with Laplace's approximation $\sqrt{2\pi n}(n/e)^n$: how does the ratio behave? Do the same for $\binom{2n}n$ computed from the circle integral of Example 11.11.

**P11.5 (★★) CS application: quicksort.** Simulate randomised quicksort on $n=100$, $1000$, $10^4$ distinct keys and compare the average number of comparisons with the exact mean $C_n=2(n+1)H_n-4n$ and with the expansion of Exercise 11.3. How large is $n^2$ times the error of the expansion?

**P11.6 (★★) The transfer theorem.** Compute exactly the number $g_n$ of 2-regular graphs on $n$ labelled vertices with the exponential formula, and compare $g_n/n!$ with $\frac{e^{-3/4}}{\sqrt{\pi n}}$ (Example 11.14, Theorem 11.13). Do the same for the Motzkin numbers (Exercise 11.6). Finally compute $p(n)$ for $n\le5000$ and compare $\ln p(n)$ with $\pi\sqrt{2n/3}$ and $p(n)$ with the Hardy–Ramanujan formula $\frac{e^{\pi\sqrt{2n/3}}}{4\sqrt3\,n}$ (Exercise 11.5).

## Solutions

### P11.1

In [ ]:
import math
import mpmath as mp
mp.mp.dps = 30

def euler_f(x):
    """f(x) = int_0^oo e^{-t}/(1+xt) dt, computed by quadrature."""
    return mp.quad(lambda t: mp.exp(-t) / (1 + x * t), [0, mp.inf])

def partial(x, K):
    return mp.fsum((-1)**k * mp.factorial(k) * x**k for k in range(K))

for x in (mp.mpf('0.1'), mp.mpf('0.05')):
    F = euler_f(x)
    errs = [(K, abs(F - partial(x, K))) for K in range(1, int(3 / x))]
    Kbest, ebest = min(errs, key=lambda e: e[1])
    print(f"x = {float(x)}: f(x) = {mp.nstr(F, 12)}")
    print("  K    error      bound K! x^K")
    for K, e in errs[::max(1, len(errs) // 8)]:
        print(f"  {K:2d}  {float(e):.3e}  {float(mp.factorial(K) * x**K):.3e}")
    print(f"  best K = {Kbest}, error {float(ebest):.3e}, "
          f"sqrt(2 pi/x) e^(-1/x) = {float(mp.sqrt(2 * mp.pi / x) * mp.exp(-1 / x)):.3e}")

In [ ]:
x = mp.mpf('0.1')
lo, hi = sorted((partial(x, 10), partial(x, 11)))
print("certified:", mp.nstr(lo, 10), "<= f(0.1) <=", mp.nstr(hi, 10), " width", mp.nstr(hi - lo, 3))
print("Euler's 0! - 1! + 2! - ... = f(1) =", mp.nstr(euler_f(1), 12))

The error first decreases, roughly like the bound $K!\,x^K$, then explodes: at $x=0.1$ the best is $K=10$, with error $1.8\cdot10^{-4}$, and $28$ terms give an error of $8$. At $x=0.05$ the best $K$ doubles to $20$ and the best error drops to $10^{-8}$, in line with $\sqrt{2\pi/x}\,e^{-1/x}$, which is about twice the actual best error. Since the error has the sign of the first omitted term, two consecutive partial sums bracket $f(x)$: an interval of width $3.6\cdot10^{-4}$ certified without computing the integral.

### P11.2

In [ ]:
from fractions import Fraction as Fr

def bernoulli_list(n):
    """B_0..B_n as exact fractions, from sum_{k<=m} C(m+1,k) B_k = 0 (Chapter 1)."""
    B = [Fr(1)]
    for m in range(1, n + 1):
        B.append(-sum(math.comb(m + 1, k) * B[k] for k in range(m)) / (m + 1))
    return B

B = bernoulli_list(120)

def poch(s, k):
    p = 1
    for i in range(k):
        p *= s + i
    return p

def zeta_em(s, N=10, M=6):
    """Proposition 11.7 without the remainder: valid for Re s > 1 - 2M, s != 1."""
    s = mp.mpc(s)
    tot = mp.fsum(mp.power(n, -s) for n in range(1, N + 1))
    tot += mp.power(N, 1 - s) / (s - 1) - mp.power(N, -s) / 2
    for k in range(1, M + 1):
        tot += mp.mpf(B[2 * k].numerator) / B[2 * k].denominator / mp.factorial(2 * k) \
               * poch(s, 2 * k - 1) * mp.power(N, 1 - s - 2 * k)
    return tot

for s in (3, 2, mp.mpc(0.5, 14.134725), mp.mpc(-2.5, 3), -1.5):
    v = zeta_em(s)
    print(f"s = {mp.nstr(s, 8):>22}:  EM {mp.nstr(v, 14):>34}   error {float(abs(v - mp.zeta(s))):.1e}")

In [ ]:
def zeta_neg_exact(n):
    """zeta(-n) from the proposition with N = 1 and 2M >= n + 1: the integral term vanishes."""
    M = n // 2 + 1
    tot = Fr(1) + Fr(1, -n - 1) - Fr(1, 2)
    for k in range(1, M + 1):
        tot += B[2 * k] / math.factorial(2 * k) * poch(-n, 2 * k - 1)
    return tot

ok = all(zeta_neg_exact(n) == -B[n + 1] / (n + 1) for n in range(1, 21)) and zeta_neg_exact(0) == Fr(-1, 2)
print("zeta(-n) = -B_{n+1}/(n+1) for n = 1..20, and zeta(0) = -1/2:", ok)
print([str(zeta_neg_exact(n)) for n in range(0, 8)])

In [ ]:
def Bf(k):
    return mp.mpf(B[k].numerator) / B[k].denominator

for n in (10, 100):
    H = mp.fsum(mp.mpf(1) / k for k in range(1, n + 1))
    print(f"n = {n}")
    for M in range(1, 6):
        approx = mp.log(n) + mp.euler + mp.mpf(1) / (2 * n) - mp.fsum(Bf(2 * k) / (2 * k * mp.mpf(n)**(2 * k)) for k in range(1, M))
        omitted = -Bf(2 * M) / (2 * M * mp.mpf(n)**(2 * M))
        err = H - approx
        print(f"   {M - 1} correction terms: error {float(err):+.3e}   first omitted term {float(omitted):+.3e}   ratio {float(err / omitted):.4f}")

Ten terms of the series plus six correction terms give $\zeta(3)$ and $\zeta(2)$ to full double precision, and $\zeta$ at the first zero to $2\cdot10^{-9}$ (the correction terms grow with $|s|$, so high on the critical line one needs larger $N$, or the Riemann–Siegel formula of the Chapter 10 notebook). At $s=-1.5$ and $-2.5+3i$ no series converges, yet the formula is accurate: the remainder integral is holomorphic for $\operatorname{Re}s>1-2M=-11$. At the negative integers the remainder vanishes identically and exact arithmetic gives the Bernoulli values of Chapter 10. For $H_n$ the error always has the sign of the first omitted term and a ratio to it between $0$ and $1$, as the corollary proves; at $n=100$ the ratio is $1$ to four digits, since the next term is $10^4$ times smaller.

### P11.3

In [ ]:
def stirling_partial(x, m):
    """(x-1/2) ln x - x + ln(2 pi)/2 + the terms k = 1..m-1."""
    s = (x - mp.mpf(1) / 2) * mp.log(x) - x + mp.log(2 * mp.pi) / 2
    for k in range(1, m):
        s += mp.mpf(B[2 * k].numerator) / B[2 * k].denominator / (2 * k * (2 * k - 1) * x**(2 * k - 1))
    return s

def T(x, m):
    return mp.mpf(B[2 * m].numerator) / B[2 * m].denominator / (2 * m * (2 * m - 1) * x**(2 * m - 1))

worst = 0
for x in [mp.mpf(k) / 2 for k in range(1, 21)]:
    for m in range(1, 12):
        r = (mp.loggamma(x) - stirling_partial(x, m)) / T(x, m)
        assert 0 < r < 1, (x, m, r)
        worst = max(worst, r)
print("0 < rho_m / T_m < 1 for x = 0.5, 1, ..., 10 and m = 1..11; largest ratio", mp.nstr(worst, 6))
for x in (1, 2, 4, 8):
    x = mp.mpf(x)
    terms = [(m, abs(T(x, m))) for m in range(1, 60)]
    mbest, tbest = min(terms, key=lambda t: t[1])
    print(f"x = {int(x)}: smallest term at m = {mbest} (pi x = {float(mp.pi * x):.2f}), "
          f"|T| = {float(tbest):.2e}, e^(-2 pi x)/(pi sqrt x) = {float(mp.exp(-2 * mp.pi * x) / (mp.pi * mp.sqrt(x))):.2e}")

In [ ]:
def loggamma_stirling(x, shift=10, m=8):
    """ln Gamma(x) for x > 0: move to x + shift with Gamma(x) = Gamma(x+n)/(x(x+1)...(x+n-1)), then Stirling."""
    x = mp.mpf(x)
    y = x + shift
    return stirling_partial(y, m) - mp.fsum(mp.log(x + j) for j in range(shift))

for x in ('0.5', '1.5', '3.7', '0.01'):
    v = loggamma_stirling(x)
    print(f"ln Gamma({x}) = {mp.nstr(v, 20)}   error {float(abs(v - mp.loggamma(mp.mpf(x)))):.1e}")
print("bound on the error: |T_8(10.5)| =", float(abs(T(mp.mpf('10.5'), 8))))

The ratio $\rho_m/T_m$ stays in $]0,1[$ everywhere, approaching $1$ for large $x$ (where the omitted terms after $T_m$ are tiny). The smallest term sits at $m\approx\pi x$ and matches $\frac{e^{-2\pi x}}{\pi\sqrt x}$ to two or three digits; for $x=8$ it is $10^{-23}$. So for $\ln\Gamma$ the divergence is harmless: moving $x$ up by $10$ costs ten logarithms and makes eight terms of the series accurate to $10^{-17}$, the bound $|T_8(10.5)|$ printed at the end. This shift-then-expand strategy is how many libraries compute $\ln\Gamma$ for large arguments.

### P11.4

In [ ]:
def laplace_check(n):
    exact = mp.factorial(n)
    integral = mp.mpf(n)**(n + 1) * mp.exp(-n) * mp.quad(lambda u: mp.exp(n * (mp.log(u) - u + 1)), [0, 0.5, 0.9, 1, 1.1, 1.5, 3, mp.inf])
    laplace = mp.sqrt(2 * mp.pi * n) * (n / mp.e)**n
    return exact, integral, laplace

print("  n   n!/(Laplace)    1 + 1/(12n)")
for n in (1, 5, 10, 50, 100):
    ex, integ, lap = laplace_check(n)
    assert abs(ex / integ - 1) < 1e-12, (n, ex / integ)
    print(f"{n:3d}   {mp.nstr(ex / lap, 10):>12}   {mp.nstr(1 + mp.mpf(1) / (12 * n), 10):>12}")
print()
print("  n   C(2n,n) from the circle integral    ratio to 4^n/sqrt(pi n)    1 - 1/(8n)")
for n in (5, 20, 100):
    circ = mp.quad(lambda t: (4 * mp.cos(t / 2)**2)**n, [-mp.pi, 0, mp.pi]) / (2 * mp.pi)
    assert abs(circ - mp.binomial(2 * n, n)) < 1e-15 * circ
    print(f"{n:3d}   {mp.nstr(circ, 18):>32}   {mp.nstr(circ / (4**mp.mpf(n) / mp.sqrt(mp.pi * n)), 10):>12}   {1 - 1 / (8 * n):.10f}")

The integral formula for $n!$ holds to the precision of the quadrature. The ratio $n!/\sqrt{2\pi n}(n/e)^n$ is close to $1+\frac1{12n}$, which is $e^{1/(12n)}$ to first order: the first term of Stirling's series. Laplace's method gives only the leading term, but the error is visibly of order $\frac1n$, as a closer look at the proof (with a third-order Taylor expansion) would show. The circle integral reproduces $\binom{2n}n$ exactly, and the ratio to $\frac{4^n}{\sqrt{\pi n}}$ follows $1-\frac1{8n}$, as Proposition 11.12 predicts.

### P11.5

In [ ]:
import random

def quicksort_comparisons(a):
    """Number of comparisons made by randomised quicksort on the list a (iterative, distinct keys)."""
    count, stack = 0, [a]
    while stack:
        b = stack.pop()
        if len(b) <= 1:
            continue
        p = b[random.randrange(len(b))]
        left = [v for v in b if v < p]
        right = [v for v in b if v > p]
        count += len(b) - 1
        stack.append(left)
        stack.append(right)
    return count

def C_exact(n):
    return 2 * (n + 1) * mp.fsum(mp.mpf(1) / k for k in range(1, n + 1)) - 4 * n

def C_asym(n):
    g = mp.euler
    return 2 * n * mp.log(n) + (2 * g - 4) * n + 2 * mp.log(n) + 2 * g + 1 + mp.mpf(5) / (6 * n)

random.seed(1)
for n, runs in ((100, 4000), (1000, 1000), (10000, 100)):
    data = [quicksort_comparisons(list(range(n))) for _ in range(runs)]
    mean = sum(data) / runs
    sd = (sum((d - mean)**2 for d in data) / (runs - 1))**0.5
    print(f"n = {n:5d}: simulated {mean:10.1f} +- {sd / runs**0.5:5.1f}   exact {float(C_exact(n)):10.1f}   "
          f"expansion {float(C_asym(n)):10.3f}   n^2 (exact - expansion) = {float((C_exact(n) - C_asym(n)) * n**2):+.4f}")

The simulated means agree with $C_n$ within about one standard error. The expansion is far more accurate than the simulation could ever tell: $n^2$ times its error is close to $-\frac16$ for every $n$, so the next term of the expansion is about $-\frac1{6n^2}$, and at $n=1000$ the expansion is right to $10^{-7}$. The leading term $2n\ln n$ alone would be off by about $2600$ at $n=1000$: the term $(2\gamma-4)n$ matters, and an analysis that stops at $O(n\log n)$ hides it.

### P11.6

In [ ]:
def egf_coefficients_exp(c, N):
    """Exact h_n = n! [z^n] exp(sum_k c_k z^k / k!) via n h_n... recurrence h_{n} = sum_{k=1}^{n} C(n-1,k-1) c_k h_{n-k}."""
    h = [1] + [0] * N
    for n in range(1, N + 1):
        h[n] = sum(math.comb(n - 1, k - 1) * c[k] * h[n - k] for k in range(1, n + 1))
    return h

N = 200
c = [0] * (N + 1)
for k in range(3, N + 1):
    c[k] = math.factorial(k - 1) // 2          # undirected cycles on k labelled vertices
g = egf_coefficients_exp(c, N)
print("2-regular graphs g_n:", g[:11])
for n in (10, 50, 100, 200):
    ratio = mp.mpf(g[n]) / mp.factorial(n) / (mp.exp(-0.75) / mp.sqrt(mp.pi * n))
    print(f"  n = {n:3d}: (g_n/n!) / (e^(-3/4)/sqrt(pi n)) = {mp.nstr(ratio, 8)}   n*(ratio-1) = {mp.nstr(n * (ratio - 1), 5)}")

Mz = [1, 1]
for n in range(2, 1001):
    Mz.append(Mz[-1] + sum(Mz[k] * Mz[n - 2 - k] for k in range(n - 1)))
print("Motzkin:", Mz[:10])
for n in (10, 100, 1000):
    print(f"  n = {n:4d}: M_n / (3^(n+3/2) / (2 sqrt(pi) n^(3/2))) = {mp.nstr(Mz[n] / (mp.mpf(3)**(n + 1.5) / (2 * mp.sqrt(mp.pi) * mp.mpf(n)**1.5)), 8)}")

In [ ]:
NP = 5000
p = [1] + [0] * NP
for k in range(1, NP + 1):
    for m in range(k, NP + 1):
        p[m] += p[m - k]
print(" n      ln p(n)/(pi sqrt(2n/3))    p(n)/HR(n)     p(n) >= 2^(floor(sqrt n)-1)")
for n in (10, 100, 1000, 5000):
    hr = mp.exp(mp.pi * mp.sqrt(2 * n / mp.mpf(3))) / (4 * mp.sqrt(3) * n)
    print(f"{n:5d}   {mp.nstr(mp.log(p[n]) / (mp.pi * mp.sqrt(2 * n / mp.mpf(3))), 6):>10}   {mp.nstr(p[n] / hr, 6):>18}   {p[n] >= 2**(math.isqrt(n) - 1)}")

For 2-regular graphs the ratio tends to $1$ with $n(\text{ratio}-1)\to-0.625$: the error is of order $\frac1n$ relative, that is $O(n^{-3/2})$ absolute, as the transfer theorem says. The Motzkin ratio converges at the same rate. For partitions, $\frac{\ln p(n)}{\pi\sqrt{2n/3}}$ creeps up slowly ($0.94$ at $n=5000$) because of the factor $\frac1{4\sqrt3\,n}$, while the full Hardy–Ramanujan formula is within $0.6\%$ at $n=5000$. The crude lower bound $2^{\lfloor\sqrt n\rfloor-1}$ holds, and is very far from the truth.